# Démo — Optimiseur de coûts opérationnels miniers

Ce notebook illustre l'architecture du projet de bout en bout, sur la chaîne
opérationnelle **forage -> sautage -> chargement -> transport -> énergie** :
1. Données synthétiques (KPI quotidiens + flotte d'équipements + panel de pannes)
2. Modèle de coûts (décomposition par étape, tendance)
3. Maintenance prédictive (risque de panne à 7 jours)
4. Optimisation (recherche opérationnelle) — arbitrage flotte propre / sous-traitance

> ⚠️ Toutes les données utilisées ici sont **100% synthétiques**, générées par `data/generate_synthetic_data.py`. Aucune donnée réelle n'est utilisée.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.cost_model import load_daily_kpi, cost_breakdown, cost_trend, month_over_month_delta, summary_kpis
from src.predictive_maintenance import load_equipment_daily, train_and_evaluate, current_risk_scores
from src.optimization import run_scenario

## 1. KPI quotidiens (production, coûts par étape, disponibilité, HSE)

In [ ]:
kpi = load_daily_kpi()
kpi.head()

In [ ]:
kpi.set_index("date")["cout_total_usd_t"].plot(title="Coût total (USD/tonne) — historique simulé", figsize=(10, 3));

## 2. Modèle de coûts — décomposition par étape

In [ ]:
breakdown = cost_breakdown(kpi, period="last_90d")
print(breakdown)
breakdown.plot.bar(title="Coût moyen par étape, 90 derniers jours (USD/tonne)", figsize=(7, 3));

In [ ]:
print(summary_kpis(kpi))
print(month_over_month_delta(kpi))

## 3. Maintenance prédictive

Régression logistique prédisant le risque de panne d'un équipement dans les 7 prochains jours, à partir de son usure (heures depuis la dernière maintenance) et de son type. Split temporel (jamais d'entraînement sur le futur).

In [ ]:
eq = load_equipment_daily()
result = train_and_evaluate(eq)
{k: v for k, v in result.items() if k != "pipeline"}

In [ ]:
risk = current_risk_scores(result["pipeline"], eq)
risk.head(10)

## 4. Optimisation — arbitrage flotte propre / sous-traitance

Programme linéaire (scipy.optimize.linprog) : pour un objectif de tonnage sur 7 jours, trouve la répartition d'heures la moins coûteuse entre flotte propre (capacité limitée) et sous-traitance (coût plus élevé, capacité non limitée), par étape.

In [ ]:
for mult in [1.0, 1.10, 1.25]:
    scenario = run_scenario(target_multiplier=mult)
    print(f"--- Objectif = {int(mult*100)}% de la production récente ({scenario.target_tonnage} t / 7j) ---")
    print(f"Coût total optimisé : {scenario.total_cost_usd:,} USD")
    print(f"Manque à gagner (flotte propre seule) : {scenario.baseline_shortfall_tonnes:,} t")
    print(f"Coût marginal de la sous-traitance : {scenario.cout_marginal_usd_par_tonne} USD/t")
    print()

In [ ]:
scenario = run_scenario(target_multiplier=1.10)
scenario.stage_allocation

## 5. Limites et pistes d'amélioration

- **Données et débits stylisés** : les débits horaires par étape (tonnes/heure) sont calibrés pour que la capacité de flotte simulée corresponde à la production moyenne du jeu de données synthétique — ce ne sont pas des ratios d'ingénierie minière réels.
- **Sautage hors optimisation de flotte** : traité comme un coût variable au tonnage, car il ne repose pas sur une flotte d'équipement avec heures d'exploitation dans ce modèle.
- **Maintenance prédictive** : modèle simple (régression logistique) à but démonstratif ; en production, on enrichirait les features (vibrations, température, historique détaillé des pannes) et testerait des modèles plus riches (gradient boosting, survie).
- **Optimisation** : modèle statique par horizon (7 jours) ; une version plus avancée intégrerait la variabilité stochastique de la disponibilité flotte (optimisation robuste / sous incertitude) plutôt que des capacités moyennes historiques.